# olmOCR-7B Test: AllenAI's Document Specialist

This notebook tests **olmOCR**, a model specifically fine-tuned for high-fidelity document parsing and Markdown conversion.

### 1. Setup

```bash
uv pip install openai pillow
```


In [1]:
import base64
import time
from openai import OpenAI
from PIL import Image
import io
import sys

# Point to the local server (LM Studio / Ollama)
client = OpenAI(base_url="http://localhost:1234/v1", api_key="lm-studio")


def encode_image(image_path):
    with open(image_path, "rb") as image_file:
        return base64.b64encode(image_file.read()).decode("utf-8")


img_path = "ollama_test_table.png"
base64_image = encode_image(img_path)

print("Setup complete. Ready to talk to olmOCR.")

Setup complete. Ready to talk to olmOCR.


### 2. Run Inference: Document-to-Markdown (STREAMING)

olmOCR is designed to turn images directly into clean Markdown. We will use a direct parsing prompt.


In [2]:
print("Requesting streaming inference from olmOCR-7B...\n")
start_time = time.time()

response = client.chat.completions.create(
    model="olmocr-7b",  # LM Studio uses the loaded model
    messages=[
        {
            "role": "system",
            "content": "Convert to text",
        },
        {
            "role": "user",
            "content": [
                {
                    "type": "image_url",
                    "image_url": {"url": f"data:image/png;base64,{base64_image}"},
                },
            ],
        },
    ],
    max_tokens=4096,
    temperature=0,
    stream=True,
)

full_response = ""
print("--- Real-time Output ---")
for chunk in response:
    if chunk.choices[0].delta.content:
        content = chunk.choices[0].delta.content
        print(content, end="", flush=True)
        full_response += content

end_time = time.time()
print(f"\n\n--- Inference Finished in {end_time - start_time:.2f} seconds ---")

Requesting streaming inference from olmOCR-7B...

--- Real-time Output ---
«الجناح»
<table>
  <tr>
    <th>النقطة الواجب خصمه</th>
    <th>الجناح</th>
    <th>الرقم الترتيب</th>
  </tr>
  <tr>
    <td>...</td>
    <td>...</td>
    <td>01</td>
  </tr>
  <tr>
    <td>...</td>
    <td>...</td>
    <td>...</td>
  </tr>
  <tr>
    <td>6</td>
    <td>سياقة مركبة تحت تأثير الكحول أو تحت تأثير المواد المخدرة - رفض الخضوع للرائت المشار إليه في المادة 207 أدناه أو للتحقيقات أو لاختبارات الكشف المنصوص عليها في المادتين 208 و213 أدناه.</td>
    <td>07</td>
  </tr>
  <tr>
    <td>...</td>
    <td>...</td>
    <td>...</td>
  </tr>
  <tr>
    <td>2</td>
    <td>السائق الذي وجه إليه الأمر بالتوقف وامتنع من تنفيذه أو لم يحترم الأمر بتوقف المركبة أو رفض سياقة مركبته أو العمل على سياقها إلى المحجز أو رفض الامتثال للأوامر القانونية الصادرة إليه</td>
    <td>13</td>
  </tr>
  <tr>
    <td>...</td>
    <td>...</td>
    <td>...</td>
  </tr>
  <tr>
    <td>...</td>
    <td>...</td>
    <td>18</td>
  </tr>
</t

### 3. Display Rendered Result


In [ ]:
from IPython.display import display, Markdown

display(Markdown(full_response))